# Session 2 — LSTM, LSTM-ARIMA Hybrid & Volatility Forecasting

**Course:** AI in Investment Strategies  
**Instructor:** Robert Ślepaczuk, WNE University of Warsaw

## Objectives
1. Feature engineering with expanding-window normalisation (no lookahead)
2. Train an LSTM with **GMADL** loss on BTC-USD
3. Build an **LSTM-ARIMA hybrid** (Kashif & Ślepaczuk, KBS 2025)
4. Volatility forecasting: GARCH vs. LSTM vs. **Hybrid VIX+GARCH+LSTM** (Roszyk & Ślepaczuk, 2024)
5. Walk-forward evaluation with MADL and Sharpe metrics

### Key references
- Kashif & Ślepaczuk (2025) — LSTM-ARIMA, *Knowledge-Based Systems* 320
- Roszyk & Ślepaczuk (2024) — Hybrid volatility, *arXiv:2407.16780*
- Michańków, Sakowski & Ślepaczuk (2024) — MADL/GMADL, *J. Comp. Sci.* 81

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
import os
# Urzadzenie: CUDA gdy dostepna. MPS (Apple Silicon) TYLKO na wyrazne zyczenie,
# bo liczy w innej kolejnosci zmiennoprzecinkowej niz CPU i te same komorki daja
# nieco inne wyniki - a material dydaktyczny ma dawac te same liczby na kazdej
# maszynie. Wlaczenie: ustaw COURSE_USE_MPS=1 przed uruchomieniem Jupytera.
DEVICE = ('cuda' if torch.cuda.is_available()
          else 'mps' if os.environ.get('COURSE_USE_MPS') and torch.backends.mps.is_available()
          else 'cpu')
COLORS = {'lstm': '#003366', 'arima': '#888888', 'hybrid': '#c41230', 'bh': '#aaaaaa'}
print(f'Device: {DEVICE}')
torch.manual_seed(42)
np.random.seed(42)

## 1. Data

In [ ]:
btc = yf.download('BTC-USD', start='2018-01-01', end='2024-12-31', auto_adjust=True)
spy = yf.download('SPY',     start='2018-01-01', end='2024-12-31', auto_adjust=True)
vix = yf.download('^VIX',    start='2018-01-01', end='2024-12-31', auto_adjust=True)

btc_close = btc['Close'].squeeze()
btc_ret   = btc_close.pct_change().dropna()

spy_close = spy['Close'].squeeze()
spy_ret   = spy_close.pct_change().dropna()

vix_close = vix['Close'].squeeze().rename('VIX')

print(f'BTC: {len(btc_ret)} days | SPY: {len(spy_ret)} days | VIX: {len(vix_close)} days')

## 2. Loss Functions: MADL & GMADL (differentiable)

In [ ]:
def madl_np(r_hat, r_actual):
    """
    MADL (JoCS 2024, eq. 6). Minimize.
    < 0 = profitable (correct directions), ~0 = random, > 0 = losing.
    """
    rh = np.asarray(r_hat, dtype=float)
    ra = np.asarray(r_actual, dtype=float)
    return float((-np.sign(rh * ra) * np.abs(ra)).mean())

def gmadl_smooth(y_hat: torch.Tensor, y: torch.Tensor, a: float = 4.0, b: float = 1.0) -> torch.Tensor:
    """
    Differentiable GMADL (arXiv:2412.18405v1, eq. 2). Minimize.
    Smooth sigmoid approximation of MADL for gradient-based training.
    a: slope parameter; b: magnitude power.
    Michankow, Sakowski & Slepaczuk (arXiv:2412.18405v1, 2024).
    doi:10.48550/arXiv.2412.18405
    """
    penalty = -1.0 * (torch.sigmoid(a * y_hat * y) - 0.5) * y.abs() ** b
    return penalty.mean()

def mse_loss(y_hat, y):
    return nn.MSELoss()(y_hat, y)

# Visualise GMADL penalty function (excluding the |R_i|^b factor)
# Correct range: negative = correct direction (good), positive = wrong direction (bad)
y_true_val = 0.02
y_hat_range = np.linspace(-0.1, 0.1, 300)
madl_penalty  = -np.sign(y_hat_range * y_true_val)                           # {-1, +1}
gmadl_penalty = -(1.0 / (1.0 + np.exp(-4.0 * y_hat_range * y_true_val)) - 0.5)  # (-0.5, +0.5)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(y_hat_range, madl_penalty,  color='gray',           label='MADL penalty (sharp, {-1,+1})', lw=1.5)
ax.plot(y_hat_range, gmadl_penalty, color=COLORS['hybrid'], label='GMADL penalty (smooth sigmoid, a=4)', lw=2)
ax.axvline(0, color='black', lw=0.8, ls='--')
ax.axhline(0, color='black', lw=0.5, ls=':')
ax.set_xlabel(r'Predicted return $\hat{R}_i$ (true $R_i = +0.02$)')
ax.set_ylabel('Penalty weight (before × $|R_i|^b$)')
ax.set_title('MADL vs GMADL penalty — negative = correct direction (good), positive = wrong (bad)')
ax.legend()
plt.tight_layout()
plt.show()

## Interpretation: MADL vs GMADL as a loss function

**MADL** (Mean Absolute Directional Loss, JoCS 2024, eq. 6):
$$\mathcal{L}_{\text{MADL}} = \frac{1}{N}\sum_{i=1}^{N} (-1)\cdot\operatorname{sign}(R_i \cdot \hat{R}_i)\cdot |R_i|$$

Interpretation — **minimise**:
- MADL **< 0**: correct directions, weighted by magnitude → **the strategy makes money**
- MADL **≈ 0**: random predictions (no directional skill)
- MADL **> 0**: wrong directions dominate → **the strategy loses money**

**GMADL** (arXiv:2412.18405v1, eq. 2) — a smooth version of MADL:
$$\mathcal{L}_{\text{GMADL}} = \frac{1}{N}\sum_{i=1}^{N} (-1)\cdot\left(\sigma(a R_i \hat{R}_i) - \tfrac{1}{2}\right)\cdot |R_i|^b$$

- Same value interpretation as MADL (minimise; < 0 = good, > 0 = bad).
- The sigmoid $\sigma(\cdot)$ replaces the non-differentiable $\operatorname{sign}(\cdot)$ → the gradient flows through the network.
- $a \to \infty, b=1$: GMADL $\to$ MADL.

**Why is GMADL better than MSE as a loss function?**
- MSE rewards magnitude accuracy and ignores direction — a model predicting ≈0 has low MSE but MADL ≈ 0 (random).
- GMADL weights every error by $|R_i|^b$: an error on a +3% day costs 3× more than on a +1% day.
- It directly aligns the model's objective with the trader's objective (P&L).

> **Note:** The gradients $\partial\mathcal{L}_{\text{GMADL}}/\partial\hat{R}_i$ are the same for $\sigma(-a\hat{R}_i R_i)$ and $\sigma(-a\hat{R}_i R_i)-0.5$ — model training does not change, only the interpretation of the loss value changes.

## 3. Feature Engineering (Expanding Window — No Lookahead)

In [ ]:
def make_features(prices: pd.Series, windows=(5, 10, 21, 63)) -> pd.DataFrame:
    """
    All features use only past information (expanding window for normalisation).
    """
    r = prices.pct_change()
    feats = pd.DataFrame(index=prices.index)

    for w in windows:
        feats[f'ret_{w}']      = r.rolling(w).mean()
        feats[f'vol_{w}']      = r.rolling(w).std()
        feats[f'ma_ratio_{w}'] = prices.rolling(w).mean() / prices - 1
        feats[f'rsi_{w}']      = _rsi(r, w)

    feats['ret_1']   = r
    feats['ret_5_5'] = r.rolling(5).mean().shift(5)  # lagged
    return feats.dropna()

def _rsi(r: pd.Series, w: int) -> pd.Series:
    up   = r.clip(lower=0).rolling(w).mean()
    down = (-r.clip(upper=0)).rolling(w).mean()
    rs   = up / (down + 1e-12)
    return 100 - 100 / (1 + rs)

X_df  = make_features(btc_close)
y_raw = btc_ret.shift(-1)           # predict NEXT day return
X_df, y = X_df.align(y_raw, join='inner', axis=0)
y = y.dropna()
X_df = X_df.reindex(y.index)

print(f'Features: {X_df.shape} | Target: {y.shape}')
print('Feature list:', list(X_df.columns))

## 4. LSTM Model (with GMADL Training Loss)

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size: int, hidden: int = 128,
                 layers: int = 2, dropout: float = 0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, layers,
                            dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(hidden)
        self.head = nn.Sequential(
            nn.Linear(hidden, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        return self.head(self.norm(out[:, -1, :])).squeeze(-1)


def make_sequences(X: np.ndarray, y: np.ndarray, seq_len: int = 20):
    Xs = np.array([X[i - seq_len:i] for i in range(seq_len, len(X))], dtype=np.float32)
    ys = y[seq_len:].astype(np.float32)
    return Xs, ys


def train_lstm(X_train: np.ndarray, y_train: np.ndarray,
               max_epochs: int = 50, lr: float = 1e-3,
               batch: int = 64, loss_fn: str = 'gmadl',
               seq_len: int = 20, patience: int = 8):
    scaler = StandardScaler().fit(X_train)
    Xn = scaler.transform(X_train)

    # Normalize targets so GMADL sigmoid gets meaningful arguments.
    # Raw daily returns (~0.01) give a*r̂*r ≈ 0.0004 → sigmoid ≈ 0.5 (flat, no signal).
    # After scaling to std≈1: a*r̂*r ≈ 4 → sigmoid ≈ 0.98 (strong gradient).
    y_std = float(np.std(y_train)) + 1e-8
    y_scaled = (y_train / y_std).astype(np.float32)

    Xs, ys = make_sequences(Xn, y_scaled, seq_len)

    # 80/20 temporal train/val split for early stopping
    n_val = max(seq_len + 1, int(0.2 * len(Xs)))
    Xs_tr, ys_tr = Xs[:-n_val], ys[:-n_val]
    Xs_val, ys_val = Xs[-n_val:], ys[-n_val:]

    dl = DataLoader(TensorDataset(torch.from_numpy(Xs_tr), torch.from_numpy(ys_tr)),
                    batch_size=batch, shuffle=True, drop_last=True)

    model = LSTMForecaster(Xs.shape[2]).to(DEVICE)
    model.y_std = y_std  # stored for rescaling predictions back to return scale
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

    best_val, best_state, no_improve = float('inf'), None, 0

    for _ in range(max_epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb)
            loss = gmadl_smooth(pred, yb) if loss_fn == 'gmadl' else mse_loss(pred, yb)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            xv = torch.from_numpy(Xs_val).to(DEVICE)
            yv = torch.from_numpy(ys_val).to(DEVICE)
            val_loss = (gmadl_smooth(model(xv), yv) if loss_fn == 'gmadl'
                        else mse_loss(model(xv), yv)).item()

        if val_loss < best_val - 1e-6:
            best_val, no_improve = val_loss, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, scaler, seq_len


def predict_lstm(model, scaler, seq_len, X_test: np.ndarray) -> np.ndarray:
    model.eval()
    Xn = scaler.transform(X_test)
    Xs = np.array([Xn[i - seq_len:i] for i in range(seq_len, len(Xn))], dtype=np.float32)
    y_std = getattr(model, 'y_std', 1.0)  # rescale back to original return scale
    with torch.no_grad():
        preds = model(torch.from_numpy(Xs).to(DEVICE)).cpu().numpy()
    return preds * y_std

## Interpretation: LSTM Architecture — Key Design Decisions

| Element | Role | Alternative |
|---------|------|-------------|
| `LayerNorm` after LSTM | Stabilises activations; important given the fat tails of financial data | BatchNorm (sensitive to small batch) |
| 2-layer head (`Linear→ReLU→Linear`) | Adds non-linearity; better approximation of complex dependencies | Direct Linear (weaker) |
| `CosineAnnealingLR` | Fast progress early, precision at the end; avoids local minima | ReduceLROnPlateau |
| Gradient clipping (`max_norm=1.0`) | Prevents gradient explosion — a classic LSTM problem | Clip by value (less stable) |
| Expanding window scaler | Eliminates lookahead bias — scaler fitted only on IS data | Full-sample scaler — **ERROR** |

**Hyperparameters to tune in WFO (in order of importance):**

1. `seq_len` (20–60 days) — how many past days the LSTM "remembers"
2. `hidden` (32–128) — model capacity; too large → overfitting
3. `dropout` (0.1–0.4) — regularisation; increase when overfitting is visible
4. `lr` (1e-4–1e-3) — learning rate; too large → instability
5. Number of epochs (10–50) — more is not always better; monitor the validation loss

## 5. ARIMA Baseline

In [ ]:
try:
    from statsmodels.tsa.arima.model import ARIMA
    STATSMODELS = True
except ImportError:
    STATSMODELS = False
    print('statsmodels not installed; skipping ARIMA')

def fit_predict_arima(train: np.ndarray, h: int = 1, order=(1, 0, 1)) -> float:
    """Fit ARIMA on train, forecast h-step ahead. Returns point forecast.
    Uses np.asarray().flat[] because statsmodels forecast() returns ndarray."""
    if not STATSMODELS:
        return 0.0
    try:
        m = ARIMA(train, order=order).fit()
        return float(np.asarray(m.forecast(h)).flat[-1])
    except Exception:
        return 0.0

print('ARIMA baseline ready.')

## 6. Walk-Forward: LSTM vs. ARIMA vs. LSTM-ARIMA Hybrid

Hybrid approach: $\hat r_{t+1} = \hat r^{\text{ARIMA}}_{t+1} + \hat\varepsilon^{\text{LSTM}}_{t+1}$  
LSTM is trained on ARIMA residuals — captures nonlinear structure not modelled by ARIMA.

In [ ]:
def run_wfo_models(X_df: pd.DataFrame, y: pd.Series,
                   is_days: int = 365, oos_days: int = 90,
                   max_epochs: int = 50, tc: float = 0.002):
    """
    Walk-forward comparison of ARIMA, LSTM, and LSTM-ARIMA hybrid.
    Returns dict of {model_name: oos_returns_series}
    """
    X = X_df.values
    y_vals = y.values
    dates  = y.index
    SEQ = 20

    results = {'arima': [], 'lstm': [], 'hybrid': []}

    for start in range(is_days, len(y) - oos_days, oos_days):
        X_tr  = X[start - is_days : start]
        y_tr  = y_vals[start - is_days : start]
        X_te  = X[start - SEQ : start + oos_days]
        y_te  = y_vals[start : start + oos_days]
        idx   = dates[start : start + oos_days]

        # ── ARIMA: iterated 1-step via append; fallback to multi-step ──────
        # append(refit=False) updates model state without re-estimating params.
        # forecast() returns ndarray in this statsmodels version — use .flat[0].
        if STATSMODELS:
            try:
                fitted     = ARIMA(y_tr, order=(1, 0, 1)).fit()
                arima_preds = []
                use_append  = True
                for i in range(oos_days):
                    fc = float(np.asarray(fitted.forecast(1)).flat[0])
                    arima_preds.append(0.0 if np.isnan(fc) else fc)
                    if use_append:
                        try:
                            fitted = fitted.append([y_te[i]], refit=False)
                        except Exception:
                            use_append = False  # disable; model stays at IS state
                arima_preds = np.array(arima_preds)
            except Exception:
                try:
                    arima_preds = np.asarray(ARIMA(y_tr, order=(1, 0, 1)).fit().forecast(oos_days))
                except Exception:
                    arima_preds = np.zeros(oos_days)
        else:
            arima_preds = np.zeros(oos_days)

        # ── LSTM (GMADL) ───────────────────────────────────────────────────
        model, scaler, seq = train_lstm(X_tr, y_tr, max_epochs=max_epochs)
        lstm_preds = predict_lstm(model, scaler, seq, X_te)
        lstm_preds = lstm_preds[-oos_days:] if len(lstm_preds) >= oos_days else lstm_preds
        n = min(len(lstm_preds), oos_days)

        # ── ARIMA residuals → LSTM-ARIMA hybrid ───────────────────────────
        if STATSMODELS:
            try:
                m_is = ARIMA(y_tr, order=(1, 0, 1)).fit()
                arima_is_preds = m_is.fittedvalues
            except Exception:
                arima_is_preds = np.zeros(len(y_tr))
        else:
            arima_is_preds = np.zeros(len(y_tr))

        residuals = y_tr - arima_is_preds
        model_res, scaler_res, seq_res = train_lstm(X_tr, residuals, max_epochs=max_epochs)
        res_preds = predict_lstm(model_res, scaler_res, seq_res, X_te)
        res_preds = res_preds[-oos_days:]

        hybrid_preds = arima_preds[:n] + res_preds[:n]

        # ── Signals & Returns ──────────────────────────────────────────────
        # tanh position sizing: position proportional to prediction confidence
        actual = y_te[:n]
        for model_name, preds in [('arima', arima_preds[:n]),
                                    ('lstm',  lstm_preds[:n]),
                                    ('hybrid', hybrid_preds)]:
            pred_std = np.std(preds) + 1e-8
            sigs   = np.tanh(preds / pred_std)        # multiplier 1.0: continuous confidence, not saturated to +/-1
            turnov = np.abs(np.diff(sigs, prepend=0.0)) / 2
            net    = sigs * actual - turnov * tc
            results[model_name].append(pd.Series(net, index=idx[:n]))

        madl_lstm   = madl_np(lstm_preds[:n], actual)
        madl_hybrid = madl_np(hybrid_preds,   actual)
        print(f'{idx[0].date()} — LSTM MADL={madl_lstm:.3f} | Hybrid MADL={madl_hybrid:.3f}')

    return {k: pd.concat(v).sort_index() for k, v in results.items() if v}


print('Starting walk-forward (takes ~5-10 min)...')
model_oos = run_wfo_models(X_df, y, max_epochs=50)

## Interpretation: Walk-Forward Optimisation for the LSTM

**WFO loop scheme:**
```
IS (365 days) | OOS (90 days) →  save results
   IS (365 days) | OOS (90 days) → ...
```
- In each iteration: the model is trained from scratch on the IS data.
- Prediction on OOS: the model has not seen these data during training.
- Concatenation of the OOS results = an unbiased equity curve.

**Why do we train the model from scratch in each window instead of just updating it?**
- Full retraining eliminates the risk of the model "memorising" an old market regime.
- In HFT practice retraining is done daily/weekly; here we simplify to quarterly windows.

**LSTM-ARIMA Hybrid Architecture:**
1. ARIMA models the linear AR(p), MA(q) structure of returns.
2. ARIMA residuals = what ARIMA failed to explain (non-linearities).
3. The LSTM is trained on the ARIMA residuals → it learns to complement ARIMA.
4. Forecast: $\hat{r}_{t+1} = \hat{r}^{\text{ARIMA}}_{t+1} + \hat{\varepsilon}^{\text{LSTM}}_{t+1}$

**Monitoring the model in production:**
- MADL_OOS **> 0** → the model has lost its directional edge; retraining is required.
- Ratio `SR_OOS / SR_IS < 0.5` → overfitting; reduce complexity or shorten seq_len.
- A sharp drop in the OOS equity curve → regime change; consider regime detection.

## 7. Results: LSTM vs. ARIMA vs. Hybrid

In [ ]:
def perf(r):
    sr = r.mean() / (r.std() + 1e-12) * np.sqrt(252)
    w  = (1 + r).cumprod()
    mdd = ((w - w.cummax()) / w.cummax()).min()
    return {'Sharpe': round(sr, 2), 'MD': f'{mdd:.1%}'}

common = model_oos['lstm'].index
bh_aligned = btc_ret.reindex(common)

print('\n--- Performance Summary (OOS, TC=20bps) ---')
for name, r in [('Buy & Hold', bh_aligned)] + list(model_oos.items()):
    m = perf(r)
    print(f'{name:<20}: Sharpe={m["Sharpe"]:>5.2f}, MD={m["MD"]}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for name, r, c in [
    ('Buy & Hold', bh_aligned, COLORS['bh']),
    ('ARIMA',      model_oos.get('arima',  bh_aligned), '#888888'),
    ('LSTM-GMADL', model_oos.get('lstm',   bh_aligned), COLORS['lstm']),
    ('Hybrid',     model_oos.get('hybrid', bh_aligned), COLORS['hybrid']),
]:
    (1 + r).cumprod().plot(ax=ax, label=name, color=c, linewidth=1.6)
ax.set_title('OOS Equity Curves — BTC-USD')
ax.set_ylabel('Cumulative Return')
ax.legend(fontsize=9)

ax2 = axes[1]
model_names = list(model_oos.keys())
sharpes     = [perf(model_oos[n])['Sharpe'] for n in model_names]
bar_colors  = [COLORS.get(n, '#555') for n in model_names]
ax2.bar(model_names, sharpes, color=bar_colors, alpha=0.85, edgecolor='white')
ax2.axhline(perf(bh_aligned)['Sharpe'], color='gray', ls='--', lw=1.2, label='B&H')
ax2.set_title('OOS Sharpe Ratio by Model')
ax2.set_ylabel('Sharpe Ratio')
ax2.legend()

plt.tight_layout()
plt.show()

## Interpretation of Results: LSTM vs ARIMA vs Hybrid

**How to read the equity curves:**
- Slope ↑ → higher ARC
- Depth of drawdown from the peak (MD) ↓ = better risk management
- Stability of the slope (low variance) → higher Sharpe ratio

**Theory vs. reality — an honest lesson:**

Theory predicts the hierarchy: Hybrid > LSTM > ARIMA > Buy & Hold. **The measured OOS results (table above) show something else:**
- **Buy & Hold has the highest Sharpe** — none of the models beat the passive benchmark on BTC.
- **The LSTM (GMADL) has a negative Sharpe and the deepest MD** — a naive directional strategy with full exposure goes bust on a volatile asset (it can be "short" during a bull run).
- **MADL ≈ 0 across all windows** → the model has no meaningful directional edge on the BTC return at t+1.

**Why this happens (and what it teaches):**
- The daily BTC return is close to unpredictable from technical features — this is not a code defect, but a real property of a liquid market (close to semi-strong efficiency).
- We checked empirically: neither vol-targeting, nor signal thresholding, nor ensembling across seeds creates an edge — **you cannot manufacture, through risk management, alpha that is not in the signal.**
- **Didactic takeaway:** a good loss function (GMADL) and a correct WFO methodology are *necessary but not sufficient*. Without a real signal the strategy does not make money. This motivates (a) the search for better predictors and (b) a risk management layer (Session 3) that limits the loss when there is no edge.

**Interpretation of MADL values (OOS, with BTC returns ~1-3%/day):**

| MADL | Interpretation |
|------|---------------|
| MADL ≪ 0 | Strong directional edge — the model profits on large moves |
| MADL < 0 | Directional edge — predictions generate profit |
| MADL ≈ 0 | No edge (random) — **our case** |
| MADL > 0 | The model loses on directional predictions — not suitable for trading |

**Impact of transaction costs:**
- A model with MADL < 0 may still have Sharpe ≈ 0 after costs given high turnover.
- Always test at 0, 10, 20 bps; the strategy should remain profitable at ≤ 10 bps.
- GMADL as a loss function indirectly penalises high turnover (errors on small moves
  matter less → the model is more cautious in generating signals).

## 8. Volatility Forecasting: GARCH vs. LSTM vs. VIX+GARCH+LSTM Hybrid

Replicates the approach from: Roszyk & Ślepaczuk (arXiv:2407.16780, 2024)

In [ ]:
try:
    from arch import arch_model
    ARCH_AVAILABLE = True
except ImportError:
    ARCH_AVAILABLE = False
    print('arch not installed; install with: pip install arch')

# ── Prepare S&P 500 returns + VIX for volatility modelling ──────────────────
spy_vix = pd.concat([spy_ret.rename('ret'), vix_close], axis=1).dropna()

REALISED_VOL_DAYS = 21   # realised vol = target (21-day rolling std)
rv = spy_vix['ret'].rolling(REALISED_VOL_DAYS).std() * np.sqrt(252)
rv.name = 'RV'

def garch_forecast(returns: pd.Series, horizon: int = 21) -> pd.Series:
    if not ARCH_AVAILABLE:
        return pd.Series(np.nan, index=returns.index)
    scaled = returns * 100  # GARCH prefers percentage returns
    forecasts = []
    dates     = []
    for end in range(252, len(returns) - horizon, 21):
        try:
            m = arch_model(scaled.iloc[:end], vol='Garch', p=1, q=1, rescale=False)
            res = m.fit(disp='off', show_warning=False)
            fc  = res.forecast(horizon=horizon)
            # Average daily variance across horizon → expected 21-day vol
            # Matches target: rv = rolling(21).std() * sqrt(252)
            avg_var = fc.variance.iloc[-1].mean()
            vol_fc  = float(np.sqrt(avg_var)) / 100 * np.sqrt(252)
            forecasts.append(vol_fc)
            dates.append(returns.index[end])
        except Exception:
            pass
    return pd.Series(forecasts, index=dates, name='GARCH_vol')

print('Fitting GARCH on S&P 500 (rolling, this may take 1-2 min)...')
garch_fc = garch_forecast(spy_vix['ret'])
print(f'GARCH forecasts: {len(garch_fc)} observations')

In [ ]:
# ── Build vol forecast features ──────────────────────────────────────────────
vix_shifted = spy_vix['VIX'].shift(1) / 100   # VIX as annualised vol expectation

# GARCH is refit every 21 days, so garch_fc is sparse. Carry each forecast
# forward to daily frequency (no look-ahead: each value uses only data up to
# its own date). Without this, dropna() collapses vol_feat to ~71 rows and the
# WFO loop range becomes empty -> all-NaN evaluation.
garch_daily = garch_fc.reindex(spy_vix.index).ffill()

vol_feat = pd.concat([
    rv.shift(1).rename('RV_lag1'),
    rv.rolling(5).mean().shift(1).rename('RV_MA5'),
    rv.rolling(21).mean().shift(1).rename('RV_MA21'),
    vix_shifted.rename('VIX'),
    garch_daily.rename('GARCH'),
], axis=1).dropna()

target_rv = rv.reindex(vol_feat.index).shift(-1)  # predict NEXT period's RV
vol_feat, target_rv = vol_feat.align(target_rv, join='inner', axis=0)
target_rv = target_rv.dropna()
vol_feat  = vol_feat.reindex(target_rv.index)
print(f'vol_feat: {vol_feat.shape} | target: {target_rv.shape}')

# ── Train LSTM on vol features ───────────────────────────────────────────────
def run_vol_wfo(vol_feat, target_rv, is_days=252, oos_days=63, epochs=15):
    lstm_preds_all, garch_preds_all, hybrid_preds_all, vix_preds_all = [], [], [], []
    actual_all = []

    X = vol_feat.values.astype(np.float32)
    y = target_rv.values.astype(np.float32)
    dates = target_rv.index
    SEQ = 20   # match train_lstm seq_len so the full OOS window is forecast

    for start in range(is_days, len(y) - oos_days, oos_days):
        X_tr = X[start - is_days : start]
        y_tr = y[start - is_days : start]
        X_te = X[start - SEQ    : start + oos_days]
        y_te = y[start          : start + oos_days]

        model, scaler, seq = train_lstm(X_tr, y_tr, max_epochs=epochs, loss_fn='mse')
        lstm_fc = predict_lstm(model, scaler, seq, X_te)[-oos_days:]

        n  = min(len(lstm_fc), oos_days)
        idx = dates[start : start + n]

        # retrieve GARCH and VIX forecasts for this window
        g  = vol_feat.reindex(idx)['GARCH'].values[:n]
        v  = vol_feat.reindex(idx)['VIX'].values[:n]
        lf = lstm_fc[:n]

        # Simple equally-weighted hybrid
        hyb = (g + v + lf) / 3

        actual_all.append(pd.Series(y_te[:n], index=idx))
        garch_preds_all.append(pd.Series(g,   index=idx))
        vix_preds_all.append(pd.Series(v,     index=idx))
        lstm_preds_all.append(pd.Series(lf,   index=idx))
        hybrid_preds_all.append(pd.Series(hyb, index=idx))

    cat = lambda lst: pd.concat(lst).sort_index() if lst else pd.Series(dtype=float)
    return {
        'actual': cat(actual_all),
        'GARCH':  cat(garch_preds_all),
        'VIX':    cat(vix_preds_all),
        'LSTM':   cat(lstm_preds_all),
        'Hybrid': cat(hybrid_preds_all),
    }

print('Running vol WFO...')
vol_results = run_vol_wfo(vol_feat, target_rv)

In [ ]:
actual       = vol_results['actual']
vol_forecasts = {k: v for k, v in vol_results.items() if k != 'actual'}

print('\n--- Volatility Forecast Evaluation (QLIKE & MSE) ---')
print(f'{"Model":<10} {"QLIKE":>8} {"MSE":>8} {"Corr":>8}')
print('-' * 36)

for name, fc in vol_forecasts.items():
    aligned = actual.reindex(fc.index)
    mse  = float(((fc - aligned)**2).mean())
    ratio = (aligned / (fc + 1e-9))**2
    qlike = float((ratio - np.log(ratio) - 1).mean())
    corr  = float(fc.corr(aligned))
    print(f'{name:<10} {qlike:>8.4f} {mse:>8.6f} {corr:>8.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
for name, c in [('GARCH', '#888'), ('VIX', '#c41230'), ('LSTM', '#003366'), ('Hybrid', '#228B22')]:
    fc = vol_forecasts[name].reindex(actual.index)
    fc.plot(ax=ax, label=name, alpha=0.7, linewidth=1.2, color=c)
actual.plot(ax=ax, label='Realised Vol', color='black', linewidth=1.5, ls='--')
ax.set_title('S&P 500 Volatility Forecasts vs. Realised')
ax.set_ylabel('Annualised Volatility')
ax.legend(fontsize=8)

ax2 = axes[1]
mse_vals = {n: float(((vol_forecasts[n].reindex(actual.index) - actual)**2).mean())
             for n in vol_forecasts}
ax2.bar(mse_vals.keys(), mse_vals.values(),
        color=['#888', '#c41230', '#003366', '#228B22'], alpha=0.85, edgecolor='white')
ax2.set_title('MSE by Model — lower is better')
ax2.set_ylabel('MSE')

plt.tight_layout()
plt.show()

## Interpretation: Volatility Forecasting — Results and Conclusions

**Metrics for evaluating volatility forecasts:**

**QLIKE** (quasi-likelihood): the standard in the GARCH literature
- `QLIKE = E[σ²/h − log(σ²/h) − 1]`  where h = forecast, σ² = realised volatility
- Asymmetric: underestimating volatility is penalised more than overestimating
- Why? Underestimating risk → positions too large → larger losses during a shock

**Model hierarchy (typical results on the S&P 500):**
1. Hybrid (VIX + GARCH + LSTM) — lowest QLIKE and MSE
   - VIX contributes forward-looking information (the market discounts future volatility)
   - GARCH models clustering (ARCH effect)
   - LSTM captures non-linearities in the residuals
2. VIX alone — surprisingly good (the market often "knows" more than the models)
3. GARCH(1,1) — good under normal conditions, weak at regime shifts
4. LSTM alone — requires a lot of data; sensitive to regime changes

**Using volatility forecasts in position sizing:**
```python
# Volatility targeting: keep risk constant
target_vol = 0.10      # 10% annualised volatility
w_t = min(target_vol / forecast_vol, 2.0) * signal_t
```
Reduces exposure ahead of high volatility; stabilises drawdown.

---

## Summary — Session 2

| Model | Application | Strength | Risk |
|-------|-------------|--------|--------|
| ARIMA | Baseline signal | Interpretable | Linearity |
| LSTM + GMADL | Directional forecast | Non-linearity, P&L-aligned loss | Overfitting |
| LSTM-ARIMA | Hybrid forecast | Complementarity | Complexity |
| VIX+GARCH+LSTM | Vol forecast | Forward-looking info | Dependence on VIX |

**3 key lessons from this session:**
1. **Expanding window normalisation** is mandatory — full-sample means lookahead bias.
2. **GMADL > MSE** as a loss function for directional strategies — but a good loss does not substitute for a missing signal.
3. **Hybrid models help where there is something to combine:** in *volatility* forecasting the VIX+GARCH+LSTM hybrid wins (lowest QLIKE), but in forecasting the *direction* of the BTC return no model beat Buy & Hold — a directional edge is hard and requires a real signal + risk management (Session 3).